<a href="https://colab.research.google.com/github/hiiamlars/master_thesis/blob/main/assw.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup



In [1]:
# @title Dependencies

# first-party
from typing import Dict, Any, Optional, Tuple, List
import random
import time
import csv
import os

# third-party
from google.colab import drive
import pandas as pd
from openai import OpenAI

In [2]:
# @title Paths and definitions

# --- folders ---

# mount GDrive
drive.mount('/content/drive')

# dataset directory
DATASET_DIR = "/content/drive/MyDrive/Thesis/dataset/"
# create dataset directory (if not already existing)
os.makedirs(DATASET_DIR, exist_ok=True)
print(f"Ensured dataset directory exists at: {DATASET_DIR}.")

# raw directory
RAW_DIR = "/content/drive/MyDrive/Thesis/raw/"
# create raw directory (if not already existing)
os.makedirs(RAW_DIR, exist_ok=True)
print(f"Ensured raw directory exists at: {RAW_DIR}.")

# create respective result path
RESULT_PATH = os.path.join(RAW_DIR, "assw_res.csv")

# create respective log path
LOG_PATH = os.path.join(RAW_DIR, "assw_log.csv")

# --- call-settings---

# secret key
OPENAI_KEY = "Set key here"

# simulation/api-call
SIMULATE = True

# hardcoded client-values
PREPROCESSING_MODEL = "gpt-4o-2024-05-13"
TEMPERATURE_DEFAULT = 0.0
MAX_TOKENS = 4000
MAX_RETRIES = 3
RETRY_DELAY = 2.0
SYSTEM_PROMPT_ASSW = """

You are given a paper submission for a top-tier Machine Learning conference.
Your goal is to identify and list the strengths and weaknesses that the paper claims about itself.
This task requires careful reading of the paper. Please follow these steps to complete the task:
1. Carefully read the entire paper submission. As you read, identify instances where the authors mention strengths or positive aspects of their research, methodology, results, or contributions. These are the strengths claimed by the paper. Also, identify instances where the authors mention limitations, weaknesses, or areas for future improvement in their work. These are the weaknesses claimed by the paper.
2. Compile your findings into two separate lists: one for strengths and one for weaknesses.
3. For each list, write each point on a separate line, keeping it concise. Add an extra blank line between each point for clarity.
4. Format your output as follows: ¡strengths claimed by the paper¿ [List each strength claimed by the paper in separate lines, with an extra blank line between each point] ¡/strengths claimed by the paper¿ ¡weaknesses claimed by the paper¿ [List each weakness claimed by the paper in separate lines, with an extra blank line between each point] ¡/weaknesses claimed by the paper¿ Important: Focus only on the strengths and weaknesses that the paper claims about itself. Do not include your own evaluation or opinion of the paper’s merits or shortcomings. Do not include the strengths and weaknesses of the baseline.
Your task is to report what the authors themselves have stated about their work’s strengths and limitations.

""" # hardcoded as Xu et al.

# --- files and columns ---

# paper text
DATASET_PAPER = "dataset_paper.parquet"

# define target column
ASSW_COL = 'parsed_text'

# define columns of result file
BASE_RES_COLMNS = ['paper_id',
                          'assw_text',
                          "assw_status",
                          "error_message",
                          "processed_timestamp",
                          'model', 'temperature', 'reasoning_effort', 'chain_of_thought']

# define columns of log file
BASE_LOG_COLUMNS = ['model',
                    'temperature',
                    'reasoning_effort',
                    'chain_of_thought',
                    'papers_processed_count',
                    'papers_succeeded_count',
                    'configuration_status']


Mounted at /content/drive
Ensured dataset directory exists at: /content/drive/MyDrive/Thesis/dataset/.
Ensured raw directory exists at: /content/drive/MyDrive/Thesis/raw/.


# Load data

In [3]:
# @title ICLR data

# full path
FULL_PATH_ICLR = DATASET_DIR + DATASET_PAPER

try:
    # load the Parquet file into a Pandas DataFrame
    df_paper_text = pd.read_parquet(FULL_PATH_ICLR)

    # check
    print("\nDataFrame Head:")
    print(df_paper_text.head())

    print("\nDataFrame Columns:")
    print(df_paper_text.columns.tolist())

# error message for file not found
except FileNotFoundError:
    print(f"ERROR: Parquet file not found at {FULL_PATH_ICLR}. Please check the path.")

# error message for file not readable
except Exception as e:
    print(f"An error occurred while reading the Parquet file: {e}")


DataFrame Head:
      paper_id                                            pdf_url  \
0  vuD2xEtxZcj  https://openreview.net/pdf/b7b54047fdaf97f5057...   
1  WoByU5W5te0  https://openreview.net/pdf/e0646a302829ac0b05b...   
2  XZRmNjUMj0c  https://openreview.net/pdf/20516df845666b5c362...   
3  cDYRS5iZ16f  https://openreview.net/pdf/043fba8d0ed8251ba2e...   
4  Sh97TNO5YY_  https://openreview.net/pdf/106dc37f05b949150cc...   

                                            abstract  \
0  In deep learning, fine-grained N:M sparsity re...   
1  We present a novel method to regularizes neura...   
2  Neural Architecture Search (NAS) is a fast-dev...   
3  Scaling transformers has led to significant br...   
4  We are interested in in silico evaluation meth...   

                                         parsed_text  \
0  INTRODUCTION Pruning Deep Neural Networks (DNN...   
1  INTRODUCTION Recently, representing a 3D scene...   
2  INTRODUCTION Neural Architecture Search (NAS) ...   
3  INTR

# Processing

In [9]:
# @title Define simulation

# define LLMClient
class LLMClient:
    """
    Handles API calls, simulation, and retry logic (adopted from supervisor's pattern).
    """

    # define initialisation setting
    def __init__(self, simulate: bool = True, api_key: str = OPENAI_KEY, seed: int = 7):
        self.simulate = simulate
        self.rng = random.Random(seed)
        self._openai = None
        self.api_key = api_key

        # initialize client if not simulating
        self._maybe_init_clients()

    # define client initalisation
    def _maybe_init_clients(self):
        """Initializes the real OpenAI client if not in simulation mode."""
        if self.simulate:
            print("INFO: LLMClient initialized in SIMULATION mode.")
            return

        if self.api_key == "placeholder for API-key":
             print("If simulation=FALSE set true API-key")

        try:
            # only OpenAI-API needed
            self._openai = OpenAI(api_key=self.api_key)
            print("INFO: LLMClient initialized for API calls.")
        except Exception as e:
            print(f"ERROR: Could not initialize OpenAI client: {e}")
            # fallback to simulation if initialization fails
            self.simulate = True

    # define actuall call
    def call(
        self,
        user_prompt: str, # adjusted per review
        temperature: int = TEMPERATURE_DEFAULT, # hardcoded as in Xu et al.
        model: str = PREPROCESSING_MODEL, # hardcoded as in Xu et al.
        system_prompt: str = SYSTEM_PROMPT_ASSW, # hardcoded as in Xu et al.
        max_tokens: int = MAX_TOKENS, # hardcoded as in Xu et al.
        max_retries: int = MAX_RETRIES, # hardcoded
        retry_delay: float = RETRY_DELAY, # hardcoded
        ) -> Tuple[str, str]:
        """
        Executes the API call with retry logic (adopted from supervisor's pattern).

        Returns: (raw_text, status_code/error_message)
        """

        # simulation path
        if self.simulate:
            simulated_text = (
                f"Simulated ASSW extraction: The core argument was '{user_prompt[:50]}...'. "
                f"Model={model}, Temp={temperature}. Output is a concise summary."
            )
            return simulated_text, "SIMULATED"

        # real API call
        for attempt in range(1, max_retries + 1):
            try:
                response = self._openai.chat.completions.create(
                    model=model,
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt}
                    ],
                    temperature=temperature,
                    max_tokens=max_tokens
                )

                # extract text and return
                generated_text = response.choices[0].message.content
                return generated_text, "SUCCESS"

            except Exception as e:

                # error message
                error_msg = f"API Error (Attempt {attempt}/{max_retries}): {type(e).__name__}: {e}"

                # after maximum retries
                if attempt == max_retries:
                    print(f"FATAL: {error_msg}")
                    return f"ERROR: Failed after {max_retries} attempts. {e}", "FAILURE"

                # otherwise retry after delay
                print(f"WARNING: {error_msg}. Retrying in {retry_delay}s...")
                time.sleep(retry_delay)

        # message after final failure
        return "ERROR: Loop finished without success.", "FAILURE"


# define review standardisation function
def create_assw(
    client: LLMClient,
    user_prompt_assw: str,
    id_variable: str,
    original_review_model: Optional[str],
    original_review_temperature: Optional[float],
    original_reasoning_effort: Optional[str] = None,
    original_chain_of_thought: Optional[str] = None,
    max_out_tokens: int = MAX_TOKENS,
    system_prompt_assw: str = SYSTEM_PROMPT_ASSW,
) -> List[Dict[str, Any]]:

    # call client as defined
    generated_text, status = client.call(
        user_prompt=user_prompt_assw, # adjusted per review
        model=PREPROCESSING_MODEL, # hardcoded as in Xu et al.
        system_prompt=SYSTEM_PROMPT_ASSW, # hardcoded as in Xu et al.
        temperature=TEMPERATURE_DEFAULT, # hardcoded as in Xu et al.
        max_tokens=MAX_TOKENS, # hardcoded as in Xu et al.
    )

    # initialize list
    records = []

    # append results to the list, including the original review's metadata
    records.append({
        'paper_id': id_variable,
        'assw_text': generated_text, # as returned
        'assw_status': status, # as returned
        'error_message': None, # as returned in case of final failure
        'processed_timestamp': time.time(), # timestamp of processing
        'model': original_review_model, # original metadata
        'temperature': original_review_temperature, # original metadata
        'reasoning_effort': original_reasoning_effort, # original metadata
        'chain_of_thought': original_chain_of_thought # original metadata
    })

    return records

# define helper function to run process_assw() on different data
def process_text_source(
    df: pd.DataFrame,
    config_model: Optional[str],
    config_temperature: Optional[float],
    config_reasoning_effort: Optional[str],
    config_chain_of_thought: Optional[str],
    results_path: str,
    log_path: str,
    text_col_name: str = ASSW_COL
):

    # information message
    print(f"\n--- Processing {text_col_name} (Model: {config_model}, Temp: {config_temperature}) ---")

    # initialize count for later messages
    papers_processed_count = 0
    papers_succeeded_count = 0

    # iterate over each paper in the provided dataframe/group
    for i, row in df.iterrows():

        # increase count
        papers_processed_count += 1

        # extract text body
        text_content = str(row[text_col_name])

        try:
            # apply create_assw
            df_result = create_assw(
                client=client,
                user_prompt_assw=text_content, # enter extracted text
                id_variable=row['paper_id'], # defined by dataframe
                original_review_model=config_model, # for logging
                original_review_temperature=config_temperature, # for logging
                original_reasoning_effort=config_reasoning_effort, # for logging
                original_chain_of_thought=config_chain_of_thought # for logging
            )

            # convert to dataframe
            df_result = pd.DataFrame(df_result)

            # adjust columns as hardcoded
            df_result = df_result[BASE_RES_COLMNS]

            # append to result file
            df_result.to_csv(results_path, mode="a", header=False, index=False)

            # succcess message
            print(f"[SUCCESS] Processed {text_col_name} for ID {row['paper_id'] if 'paper_id' in row else row['document_id']} with original config (Model: {config_model}, Temp: {config_temperature}).")

            # increase count
            papers_succeeded_count += 1

        except Exception as e:
            # create artificial error record
            failure_record = {
                'paper_id': row['paper_id'],
                'assw_text': None,
                'assw_status': "FAILURE",
                'error_message': f"Failed to process. Error: {type(e).__name__}: {str(e)}",
                'processed_timestamp': time.time(),
                'model': config_model, # for logging
                'temperature': config_temperature, # for logging
                'reasoning_effort': config_reasoning_effort, # for logging
                'chain_of_thought': config_chain_of_thought # for logging
            }

            # convert to dataframe
            df_result = pd.DataFrame([failure_record])

            # adjust columns as hardcoded
            df_result = df_result[BASE_RES_COLMNS]

            # append to result file anyway
            df_result.to_csv(results_path, mode="a", header=False, index=False)

            # failure message
            print(f"[FAILURE] Failed to process {text_col_name} for ID {row['paper_id'] if 'paper_id' in row else row['document_id']} with original config (Model: {config_model}, Temp: {config_temperature}). Error: {e}")

    # create log directory
    log_entry = {
        'model': config_model, # as in df_result
        'temperature': config_temperature, # as in df_result
        'reasoning_effort': config_reasoning_effort, # as in df_result
        'chain_of_thought': config_chain_of_thought, # as in df_result
        'papers_processed_count': papers_processed_count, # as result of iterative count
        'papers_succeeded_count': papers_succeeded_count, # as result of iterative count
        'configuration_status': "SUCCESS" if papers_succeeded_count == papers_processed_count else "PARTIAL_FAILURE" if papers_succeeded_count > 0 else "TOTAL_FAILURE"
    }

    # convert to dataframe
    log_df = pd.DataFrame([log_entry])

    # append to log file
    log_df[BASE_LOG_COLUMNS].to_csv(log_path, mode="a", header=False, index=False)

    # information message
    print(f"Logged status for {text_col_name} (Model: {config_model}, Temp: {config_temperature}): {log_entry['configuration_status']} (Processed {papers_succeeded_count}/{papers_processed_count} papers).")

In [ ]:
# @title ASSW

# initialize the client
client = LLMClient(simulate=SIMULATE)

# create new result file with hardcoded columns
if not os.path.exists(RESULT_PATH):
    with open(RESULT_PATH, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(BASE_RES_COLMNS)

# create new log file with hardcoded columns
if not os.path.exists(LOG_PATH):
    with open(LOG_PATH, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(BASE_LOG_COLUMNS)

# group by paper_id (to log each paper after the other)
grouped_human_papers = df_paper_text.groupby('paper_id')

# across all paper_id-groups
for paper_id, paper_group_df in grouped_human_papers:
    process_text_source(
        df=paper_group_df, # single paper
        text_col_name=ASSW_COL, # hardcoded
        config_model="human", # plaeholder for human reviews
        config_temperature=None, # plaeholder for human reviews
        config_reasoning_effort=None, # plaeholder for human reviews
        config_chain_of_thought=None, # plaeholder for human reviews
        results_path=RESULT_PATH, # result path
        log_path=LOG_PATH # log path
    )